<a href="https://colab.research.google.com/github/yaranoun/ML-Tech/blob/main/notebooks/01_chunk_documents.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [39]:
!git clone https://github.com/yaranoun/ML-Tech.git

Cloning into 'ML-Tech'...
remote: Enumerating objects: 159, done.
remote: Counting objects: 100% (159/159), done.
remote: Compressing objects: 100% (153/153), done.
remote: Total 159 (delta 84), reused 6 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (159/159), 70.01 KiB | 1.13 MiB/s, done.
Resolving deltas: 100% (84/84), done.


In [40]:
from pathlib import Path

%cd ML-Tech
raw_folder = Path("data/raw")

documents = []

for file_path in raw_folder.glob("*.txt"):
    with open(file_path, "r", encoding="utf-8") as f:
        text = f.read()

    documents.append({
        "filename": file_path.name,
        "content": text
    })

print(f"Loaded {len(documents)} documents")
for doc in documents:
    print(doc["filename"])

/content/ML-Tech/ML-Tech/ML-Tech/ML-Tech/ML-Tech
Loaded 6 documents
Personal attendance required.txt
Passport of adopted, born in special circumstances child or a citizen without a family name.txt
Certifying Passport.txt
Ex-porting Biometric Passport.txt
Lost Passport or Stolen Passport.txt
Biometric Passport.txt


In [41]:
document_sections = {
    "Biometric Passport.txt": [
        "Requested documents:",
        "Remarks:",
        "Fees:",
        "NB:"
    ],

    "Lost Passport or Stolen Passport.txt": [
        "Lost Passport:",
        "Stolen Passport:",
        "NB:"
    ],

    "Ex-porting Biometric Passport.txt": [
        "Exporting a Lebanese passport",
        "Exporting a Foreign passport",
        "Lebanese or foreign passport shipped:",
        "For travel agencies that plan to ship passports:",
        "For the individual planning on shipping his passport with another traveler:",
        "Nb:"
    ],
    "Passport of an adopted, born in special circumstances child or a citizen without a family name.txt":[
        "The requested documents",
        "NB:",
        "Child born in special cirmustances",
        "A passport for a minor:",
        "A citizen without a family name"
    ],
    "Personal attendance required.txt":[
        "Personal attendance required",
        "Exemption from attendance",
        "Exemption from fees"
    ],
    "Certifying Passport.txt":[]
}

In [42]:
def extract_metadata(text):
    metadata = {
        "url": "",
        "title": "",
        "category": "",
        "keywords": ""
    }

    # Separate metadata from actual content
    if "Content:" in text:
        metadata_part, content = text.split("Content:", 1)
    else:
        metadata_part = ""
        content = text

    # Extract each metadata field
    for line in metadata_part.splitlines():
        line = line.strip()

        if line.startswith("URL:"):
            metadata["url"] = line.replace("URL:", "", 1).strip()

        elif line.startswith("Title:"):
            metadata["title"] = line.replace("Title:", "", 1).strip()

        elif line.startswith("Category:"):
            metadata["category"] = line.replace("Category:", "", 1).strip()

        elif line.startswith("Keywords:"):
            metadata["keywords"] = line.replace("Keywords:", "", 1).strip()

    return metadata, content.strip()

In [43]:
metadata, content = extract_metadata(documents[0]["content"])

print(metadata)
print(content)

{'url': 'https://www.general-security.gov.lb/en/posts/73', 'title': 'Personal attendance required', 'category': 'Personal attendance required', 'keywords': 'Lebanese citizens, minors, exemption from attendance, exemption from fees'}
Personal attendance required

Lebanese citizens that request a new passport should show up personally at the competent regional center of general security, according to their place of residence, having in hand an application that’s been filled, and certified by the competent mayor.
Minors aged 7 years or younger have to accompany their parents to the mayor’s office, but don’t have to show up at the general security center. Both parents should sign a letter of consent at the mayor’s office, and convey their request to the general security. One of the parents can go on his own to the general security office, if the other parent signed the letter at the mayor’s office.
Minors aged between 7 and 15 years old, have to show up at the general security with their p

In [44]:
import re

def chunk_document(filename, text, headings,metadata):
    chunks = []

    # If there are no headings, keep the whole document
    if not headings:
        chunks.append({
            "document": filename,
            "title":metadata["title"],
            "url":metadata["url"],
            "category":metadata["category"],
            "keywords":metadata["keywords"],
            "section": "Full Document",
            "text": text.strip()
        })
        return chunks

    # Build a regex from the headings
    pattern = "|".join(re.escape(h) for h in headings)

    # Split while keeping the headings
    parts = re.split(f"({pattern})", text)

    current_heading = None
    current_text = ""

    for part in parts:

        if part in headings:

            if current_heading is not None:
                chunks.append({
                    "document": filename,
                     "title":metadata["title"],
                    "url":metadata["url"],
                    "category":metadata["category"],
                    "keywords":metadata["keywords"],
                    "section": current_heading,
                    "text": current_text.strip()
                })

            current_heading = part.rstrip(":")
            current_text = ""

        else:
            current_text += part

    # Save the last chunk
    if current_heading is not None:
        chunks.append({
            "document": filename,
            "title":metadata["title"],
            "url":metadata["url"],
            "category":metadata["category"],
            "keywords":metadata["keywords"],
            "section": current_heading,
            "text": current_text.strip()

        })

    return chunks

In [45]:
all_chunks = []
for document in documents:
    filename = document["filename"]
    raw_text = document["content"]

    # remove URL, title, category, keywords, etc.
    metadata,clean_text = extract_metadata(raw_text)

    headings = document_sections.get(filename, [])

    chunks = chunk_document(
        filename,
        clean_text,
        headings,
        metadata
    )

    all_chunks.extend(chunks)

In [46]:
import json
import os

os.makedirs("data/processed", exist_ok=True)

with open("data/processed/chunks.json", "w", encoding="utf-8") as f:
    json.dump(all_chunks, f, indent=4, ensure_ascii=False)

In [47]:
import json

with open("data/processed/chunks.json", "r", encoding="utf-8") as f:
    chunks = json.load(f)

print(json.dumps(chunks, indent=4, ensure_ascii=False))

[
    {
        "document": "Personal attendance required.txt",
        "title": "Personal attendance required",
        "url": "https://www.general-security.gov.lb/en/posts/73",
        "category": "Personal attendance required",
        "keywords": "Lebanese citizens, minors, exemption from attendance, exemption from fees",
        "section": "Personal attendance required",
        "text": "Lebanese citizens that request a new passport should show up personally at the competent regional center of general security, according to their place of residence, having in hand an application that’s been filled, and certified by the competent mayor.\nMinors aged 7 years or younger have to accompany their parents to the mayor’s office, but don’t have to show up at the general security center. Both parents should sign a letter of consent at the mayor’s office, and convey their request to the general security. One of the parents can go on his own to the general security office, if the other parent